In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

문제 1 (단답형 주관식)

이진 로지스틱 회귀 모델(Binary Logistic Regression)이 선형 회귀(Linear Regression) 모델과 구별되는 가장 큰 차이점은 '활성화 함수'입니다.

이진 분류 모델에서 시그모이드(Sigmoid) 함수가 반드시 필요한 이유와 그 주요 역할을 1~2줄로 간략히 서술하시오.

1번 답 - 시그모이드 함수는 선형 회귀의 출력을 0~1 확률로 변환하여 이진 분류가 가능하게 해준다.

문제 2 (단답형 주관식)

MSE(평균 제곱 오차)를 사용했던 것과 달리, '이진 분류' 문제에서는 BCELoss(교차 엔트로피)를 사용했습니다.

MSE가 아닌 BCELoss를 분류 문제에 사용하는 주된 이유를 서술하시오.

(힌트: 강의자료 9페이지의 학생 A, B 비유 참고)

2번 답 - 교차 엔트로피는 높은 확신을 가지고 틀렸을 때 훨신 더 큰 패널티를 부여하므로 이진분류 문제 에서는 BCELoss(교차 엔트로피)를 사용한다.

문제 3 (실습 문제 - 코드 작성)

nn.Module을 상속받아, 이진 로지스틱 회귀 모델 클래스
MyBinaryClassifier를 정의하시오.

[요구사항]

__init__ 메소드에서 n_input과 n_output을 인자로 받아, nn.Linear 계층(이름: self.linear)과 nn.Sigmoid 계층(이름: self.sigmoid)을 정의해야 합니다.

forward 메소드에서 입력 x가 self.linear와 self.sigmoid를 순서대로 통과한 최종 결과를 반환해야 합니다.

In [11]:
import torch
import torch.nn as nn

class MyBinaryClassifier(nn.Module):
    def __init__(self, n_input, n_output):
        # TODO: 1. 부모 클래스의 __init__ 호출
        super(MyBinaryClassifier,self).__init__()

        # TODO: 2. self.linear 라는 이름으로 nn.Linear 계층 정의
        self.linear = nn.Linear(n_input,n_output)

        # TODO: 3. self.sigmoid 라는 이름으로 nn.Sigmoid 계층 정의
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # TODO: 4. linear 계층과 sigmoid 계층을 순서대로 통과한 예측값 반환
       return self.sigmoid(self.linear(x))

# --- 테스트 코드 (수정 불필요) ---
# 입력 특성 2개, 출력 특성 1개
net = MyBinaryClassifier(n_input=2, n_output=1)
print(net)

# (배치 크기 2, 특성 2)의 더미 입력으로 테스트
dummy_input = torch.randn(2, 2)
output = net(dummy_input)
print(f"\n입력 크기: {dummy_input.shape}")
print(f"출력 크기: {output.shape}")
print(f"출력 값 (0~1 사이): {output.data}")

MyBinaryClassifier(
  (linear): Linear(in_features=2, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

입력 크기: torch.Size([2, 2])
출력 크기: torch.Size([2, 1])
출력 값 (0~1 사이): tensor([[0.2653],
        [0.4973]])


문제 4 (실습 문제 - 코드 빈칸 채우기)

이진 분류 모델의 학습을 위해 모델, 손실 함수, 옵티마이저를 정의하는 코드입니다.

3개의 빈칸 (# TODO: ...)을 채워 8차시 학습 내용에 맞는 설정을 완성하시오. (단, 모델은 문제 3에서 정의한 MyBinaryClassifier를 사용합니다.)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 설정값 (수정 불필요) ---
n_input = 2  # 입력 특성 수 (예: 꽃받침 길이, 너비)
n_output = 1 # 출력 특성 수 (예: 1일 확률)
learning_rate = 0.01

# --- 모델 클래스 (문제 3의 정답) ---
class MyBinaryClassifier(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.linear = nn.Linear(n_input, n_output)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))
# -----------------------------------

# TODO: 1. MyBinaryClassifier 클래스를 사용하여 n_input, n_output에 맞는 모델 'net' 인스턴스 생성
net = MyBinaryClassifier(n_input, n_output)

# TODO: 2. 이진 분류에 적합한 '교차 엔트로피' 손실 함수 'criterion' 생성 (BCELoss 사용)
criterion = nn.BCELoss()

# TODO: 3. 'SGD' 옵티마이저 'optimizer' 생성
# (net의 파라미터와 learning_rate를 인자로 전달)
optimizer = optim.SGD(net.parameters(),lr=0.01)


# --- 결과 확인 (수정 불필요) ---
print(f"모델:\n{net}")
print(f"\n손실 함수:\n{criterion}")
print(f"\n옵티마이저:\n{optimizer}")

모델:
MyBinaryClassifier(
  (linear): Linear(in_features=2, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

손실 함수:
BCELoss()

옵티마이저:
SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


문제 5 (실습 문제 - 코드 빈칸 채우기)

모델의 정확도(Accuracy)를 계산하는 코드입니다.

outputs는 시그모이드 함수를 통과한 확률값 (0~1 사이)입니다.

3개의 빈칸 (# TODO: ...)을 채워 정확도를 올바르게 계산하시오.

In [13]:
import torch

# --- 가상의 모델 출력값과 정답 (수정 불필요) ---
# outputs: 모델이 예측한 확률 (0~1 사이)
outputs = torch.tensor([0.1, 0.9, 0.4, 0.7])
# labels: 실제 정답 (0 또는 1)
labels = torch.tensor([0.0, 1.0, 1.0, 1.0])
# ------------------------------------------------

# TODO: 1. outputs의 확률값을 0.5 기준으로 0 또는 1의 'predicted' 값으로 변환
# (예: 0.5보다 크면 1, 작거나 같으면 0)
predicted = torch.where(outputs > 0.5,1,0)

# TODO: 2. 'predicted'와 'labels'가 일치하는 건수(True의 개수) 'correct_predictions' 계산
correct_predictions = (predicted == labels).sum()

# TODO: 3. 'correct_predictions'를 전체 건수로 나누어 'accuracy' 계산
accuracy = correct_predictions.float()/len(labels)


# --- 결과 확인 (수정 불필요) ---
# (정답: 0.1->0, 0.9->1, 0.4->0, 0.7->1)
# (실제: 0, 1, 1, 1)
# (결과: 0(O), 1(O), 0(X), 1(O) => 3/4 = 0.75)
print(f"예측 (0/1): {predicted}")
print(f"정답 (0/1): {labels}")
print(f"맞춘 건수: {correct_predictions}")
print(f"정확도: {accuracy:.2f}")

예측 (0/1): tensor([0, 1, 0, 1])
정답 (0/1): tensor([0., 1., 1., 1.])
맞춘 건수: 3
정확도: 0.75


문제 6 (실습 문제 - 코드 빈칸 채우기)

nn.BCELoss 대신, 수치적으로 더 안정적인 nn.BCEWithLogitsLoss를 사용하는 코드입니다.

이 손실 함수를 사용하기 위한 모델 정의와 예측 방식의 빈칸 4개를 채우시오.


In [14]:
import torch
import torch.nn as nn

# --- BCEWithLogitsLoss용 모델 정의 ---
class NetWithLogits(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        # TODO: 1. self.linear 라는 이름으로 nn.Linear 계층 정의
        # (Sigmoid 계층은 정의하지 않음)
        self.linear = nn.Linear(n_input,n_output)

    def forward(self, x):
        # TODO: 2. linear 계층만 통과한 원본 출력(logit)을 반환
        logit = self.linear(x)
        return logit

# --- 설정 (수정 불필요) ---
net_logits = NetWithLogits(2, 1)
dummy_inputs = torch.randn(5, 2)
# ----------------------------

# TODO: 3. 'BCEWithLogitsLoss' 손실 함수 'criterion_logits' 생성
criterion_logits = nn.BCEWithLogitsLoss()

# --- 모델 예측 (수정 불필요) ---
# (Sigmoid를 거치지 않은 logit 값이 출력됨)
logit_outputs = net_logits(dummy_inputs)
# ----------------------------

# TODO: 4. logit_outputs 값을 0.0 기준으로 0 또는 1의 'predicted_logits' 값으로 변환
# (BCEWithLogitsLoss는 0.5가 아닌 0.0을 기준으로 판단함)
predicted_logits = torch.where(logit_outputs > 0.0,1,0)


# --- 결과 확인 (수정 불필요) ---
print(f"손실 함수:\n{criterion_logits}")
print(f"\n모델 원본 출력 (Logits):\n{logit_outputs.data.squeeze()}")
print(f"\n최종 예측 (0/1):\n{predicted_logits.data.squeeze()}")

손실 함수:
BCEWithLogitsLoss()

모델 원본 출력 (Logits):
tensor([-0.8829, -0.0143,  0.1289, -3.1125, -2.1674])

최종 예측 (0/1):
tensor([0, 0, 1, 0, 0])


자율 과제 [라이브 코딩]

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

x_org, y_org = iris.data, iris.target

print('--- 1. 데이터 불러오기 결과 ---')
print('원본 데이터', x_org.shape, y_org.shape) #(150, 4) (150,)

In [ ]:
x_data = iris.data[:100,:2]
y_data = irirs.data[:100]

print('--- 2. 데이터 추출 결과 ---')
print('대상 데이터', x_data.shape, y_data.shape

In [1]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, train_size=70, test_size=30, random_state=123)


print('--- 3. 데이터 분할 결과 ---')
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

NameError: name 'x_data' is not defined

In [ ]:
class Net(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)
        self.sigmoid = nn.Sigmoid()

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.sigmoid(x1)
        return x2

In [ ]:
creiterion = nn.BCELoss()

lr = 0.01
optimizer = optim.SGD(net.nn.parameters(),lr=lr)

In [ ]:
inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).float()

labels1 = labels.view((-1,1))

inputs_test = torch.tensor(x_test).float()
labels_test = torch.tensor(y_test).float()

labels1_test = labels_test.view((-1,1))

In [ ]:
optput = net(inputs)

loss = criterion(outputs,labels1)

g = make_dot(loss,params = dict(net.named_parameters()))
display(g)

In [ ]:
lr = 0.01
net = Net(n_intput,n_output)
criterion = nn.BCELss()
optimizer = optim.SGD(net.parameters(),lr=lr)
num_epochs = 10000
history = np.zeros((0,5))


In [ ]:
for each in range(num_epochs):
  optimize.zero_grad()
  outputs = net(inputs)
  loss = criterion(outputs,labels1)
  loss.backward()
  optimizer.step()

  train_loss = loss.item()
  predicted = torch.where(outputs<0.5,0,1)
  train_acc = (predicted == labels1).sum()/len(y_train)

  ouputs_test = net(inputs_test)
  loss_test = criterion(outputs_test, labels1_test)
  val_loss = loss_test.item()
  predicted_test = torch.where(outputs_test < 0.5, 0, 1)
  val_acc = (predicted_test == labels1_test).sum() / len(y_test)


  if (epoch % 1000 == 0):
       print(f'Epoch [{epoch}/{num_epochs}], loss: {train_loss:.5f}, acc: {train_acc:.5f}, val_loss: {val_loss:.5f}, val_acc: {val_acc:.5f}')


  item = np.array([epoch, train_loss, train_acc, val_loss, val_acc])
  history = np.vstack((history, item))
